## Create NACC train and test splits

Created by: Sahana Kowshik

Date: 05/06/2025

In [1]:
import pandas as pd
import json

In [5]:
directory_path = "/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/nacc"
data = pd.read_csv(f"{directory_path}/NACC_wjson.csv")
nacc_np = pd.read_csv(f'/projectnb/vkolagrp/datasets/NACC/csv/processed/investigator_ftldlbd_merged_neuropath_vnum_unique_nacc65.csv')

/scratch/9210280.1.cbm.q/ipykernel_2349703/1205421490.py:2: DtypeWarning: Columns (20,22,24,26,28,40,43,45,47,50,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,164,175,178,216,219,221,223,225,227,229,231,233,235,237,239,241,243,245,247,249,251,253,255,257,259,261,263,265,267,269,271,396,398,400,418,420,422,431,444,453,492,571,599,607,663,679,696,699,716,727,733,808,819,823,825,831,892,947,948,949,957,958,959,960,970,992,995,998,1017,1022,1192,1196,1199,1395,1397,1399,1400,1402,1409,1411,1413,1414,1421,1423,1425,1427,1428,1435,1450,1464,1478,1492,1494,1530,1546,1548,1550,1552,1554,1556,1558,1560,1562,1564,1566,1568,1570,1572,1574,1576,1578,1580,1582,1584,1586,1588,1590,1592,1594,1596,1598,1600,1650,1651,1653,1654,1657,1658,1661,1662,1665,1666,1669,1670,1744,1803,1812,1814,1816,1818,1829,1831,1833,1841,1843,1845,1847,1855,1857,1859,1861,1887) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv(f"{directory_path}/NACC_

In [11]:
train_latest = data[~data['NACCID'].isin(nacc_np['NACCID'])].reset_index(drop=True)
train_latest = train_latest[train_latest['NACCUDSD'] != 2].reset_index(drop=True)

np_latest = data[data['NACCID'].isin(nacc_np['NACCID'])].reset_index(drop=True)
np_latest = np_latest[np_latest['NACCUDSD'] != 2].reset_index(drop=True)

In [12]:
print(len(train_latest))
print(len(np_latest))

41064
7812


In [13]:
np_latest['DATSCAN'].value_counts()

DATSCAN
8.0    2716
0.0      54
1.0      22
Name: count, dtype: int64

## Add some neuropath cases to training

In [14]:
np_latest['VISITGAP'] = np_latest['NACCYOD'] - np_latest['VISITYR']

In [15]:
# len(np_latest[(np_latest['NACCUDSD'] != 1) & (np_latest['NACCLEWY'].isin([0,1,2,3,4])) & (np_latest['NPFTDTAU'].isin([0,1]) & np_latest['NPFTDTDP'].isin([0,1])) & (np_latest['NPADNC'].isin([0,1,2,3])) & (np_latest['VISITGAP'] <= 3)].sample(frac=1, random_state=42).reset_index(drop=True))

In [16]:
len(np_latest[
    (np_latest['NACCUDSD'] != 1) &
    (np_latest['NACCLEWY'].isin([0, 1, 2, 3, 4])) &
    (
        (np_latest['NPFTDTAU'].isin([0]) & np_latest['NPFTDTDP'].isin([0])) | 
        np_latest['NPFTDTAU'].isin([1]) | 
        np_latest['NPFTDTDP'].isin([1])
    ) &
    (np_latest['NPADNC'].isin([0, 1, 2, 3])) &
    (np_latest['VISITGAP'] <= 3)
])

2668

In [17]:
np_df = np_latest[
    (np_latest['NACCUDSD'] != 1) &
    (np_latest['NACCLEWY'].isin([0, 1, 2, 3, 4])) &
    (
        (np_latest['NPFTDTAU'].isin([0]) & np_latest['NPFTDTDP'].isin([0])) | 
        np_latest['NPFTDTAU'].isin([1]) | 
        np_latest['NPFTDTDP'].isin([1])
    ) &
    (np_latest['NPADNC'].isin([0, 1, 2, 3])) &
    (np_latest['VISITGAP'] <= 3)
].sample(n=2000, random_state=42).reset_index(drop=True)

print(len(np_df))


2000


In [18]:
set(np_df['NACCID']).intersection(set(train_latest["NACCID"]))

set()

In [19]:
combined_train = pd.concat([train_latest, np_df], axis=0).sample(frac=1, random_state=42).reset_index(drop=True)
remaining_test = np_latest[~np_latest['NACCID'].isin(combined_train['NACCID'])].sample(frac=1, random_state=42).reset_index(drop=True)

In [20]:
combined_train['VISITGAP'].max()

3.0

In [21]:
combined_train['VISITGAP'].value_counts()


VISITGAP
1.0    806
0.0    556
2.0    374
3.0    264
Name: count, dtype: int64

In [22]:
print(f"Length of training data: {len(combined_train)}")
print(f"Length of testing data: {len(remaining_test)}")

Length of training data: 43064
Length of testing data: 5812


In [23]:
assert len(combined_train['NACCID']) + len(remaining_test['NACCID']) == len(data[data['NACCUDSD'] != 2])

In [18]:
combined_train.to_csv(f"{directory_path}/train.csv", index=False)
remaining_test.to_csv(f"{directory_path}/test.csv", index=False)